In [1]:
# %% [markdown]
# # GlyphMatics v10+D — Iterative Local Search
#
# **在 v10 基础上的唯一改动：**
#
# ### D. 迭代局部搜索（Iterative Local Search）
#
# v10 的双阶段重建（Phase 1 QR+core SVD，Phase 2 降 gain 重试）本质上都是线性投影，
# 无法跳出初始 PairFold 子空间的局部极值。
#
# v10+D 在重建完成后追加一个梯度无关的坐标扰动搜索：
# - 对 B 的每一列（rank 个方向）逐一施加随机扰动
# - 接受降低目标函数（Frobenius 残差 + RowGuard overshoot）的扰动
# - 用 lstsq 重新求最优 A，保持 delta 近似
# - 多轮迭代，step 随迭代衰减
#
# 触发条件：仅对重建被"rejected"的层触发（即 pairfold_diagonal fallback 层），
# 这些层是 v10 最薄弱的部分，搜索性价比最高。
# accepted 层不触发（已经很好了，避免破坏）。
#
# 新增参数：
# - `ILS_ENABLED`:      总开关（默认 True）
# - `ILS_N_ITER`:       迭代轮数（默认 6，每轮遍历所有 rank 列）
# - `ILS_STEP_INIT`:    初始扰动步长（默认 0.025）
# - `ILS_STEP_DECAY`:   每轮步长衰减系数（默认 0.75）
# - `ILS_ON_ACCEPTED`:  是否对 accepted 层也触发（默认 False）
#
# 其余所有逻辑与 v10 完全一致。

# %% [code]
from pathlib import Path
import os, sys, json, shutil, subprocess, importlib.util, hashlib, zipfile, time
from collections import Counter

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Python:", sys.version)
print("Working dir:", Path.cwd())

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
print("\n[Inputs]")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p)

# %% [code]
import importlib.metadata as md

def list_wheel_dirs():
    rows = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]:
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            wheels = sorted(d.glob("*.whl"))
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        low = " ".join(n.lower() for n in names)
        score = sum(1 for t in ["tinker_cookbook", "tinker-cookbook", "tinker", "chz"] if t in low)
        if score:
            candidates.append((score, d))
    candidates.sort(reverse=True)
    return candidates[0][1] if candidates else None

print("[Tinker] scanning wheel folders...")
for d, names in list_wheel_dirs():
    interesting = [n for n in names if "tinker" in n.lower() or "chz" in n.lower()]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting:
            print(" -", name)

if importlib.util.find_spec("tinker_cookbook") is None:
    wheel_dir = find_tinker_wheelhouse()
    if wheel_dir is None:
        raise FileNotFoundError("Cannot find local tinker wheelhouse.")
    print("[Tinker] selected wheel_dir:", wheel_dir)
    cmd = [sys.executable, "-m", "pip", "install", "--no-index",
           f"--find-links={wheel_dir}", "tinker-cookbook", "tinker", "chz"]
    subprocess.run(cmd, check=True)
else:
    print("[Tinker] already installed; skipping")

import tinker_cookbook
from tinker_cookbook import weights
print("[Tinker] ready:", tinker_cookbook.__file__)
for pkg in ["tinker-cookbook", "tinker", "chz"]:
    try:
        print(f"[Tinker]  {pkg}:", md.version(pkg))
    except Exception:
        pass
assert hasattr(weights, "build_lora_adapter")

# %% [code]
def find_first_existing(paths):
    for p in paths:
        q = Path(p)
        if q.exists():
            return q
    return None

ADAPTER_PATH = find_first_existing([
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter",
])

BASE_MODEL_PATH = find_first_existing([
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
])

if ADAPTER_PATH is None:
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if (folder / "adapter_model.safetensors").exists():
            ADAPTER_PATH = folder
            break

if BASE_MODEL_PATH is None:
    for cfg in Path("/kaggle/input").rglob("config.json"):
        folder = cfg.parent
        s = str(folder).lower()
        if "nemotron" in s and ("30b" in s or "nano" in s) and "adapter" not in s:
            BASE_MODEL_PATH = folder
            break

if ADAPTER_PATH is None:
    raise FileNotFoundError("Nemotron adapter not found.")
if BASE_MODEL_PATH is None:
    raise FileNotFoundError("Base model not found.")

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

assert (ADAPTER_PATH / "adapter_config.json").exists()
assert (ADAPTER_PATH / "adapter_model.safetensors").exists()
assert (BASE_MODEL_PATH / "config.json").exists()

# %% [code]
# ════════════════════════════════════════════════════════════════════════════
# GlyphMatics v10+D — Iterative Local Search
# ════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
from collections import Counter
import json, os, hashlib, math
import torch
import tinker_cookbook.weights._adapter as A

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ─── Competition knobs（v10 原有，全部保留）──────────────────────────────────
FORCED_FUSED_RANK          = int(os.environ.get("FORCED_FUSED_RANK", "32"))
SVD_ENERGY_GAIN_CAP        = float(os.environ.get("SVD_ENERGY_GAIN_CAP", "1.22"))
ROW_NORM_GUARD_ENABLED     = True
ROW_NORM_GAIN_CAP          = float(os.environ.get("ROW_NORM_GAIN_CAP", "1.10"))
PAIRFOLD_ENABLED           = True
PAIRFOLD_PAIR_WIDTH        = 2
PAIRFOLD_TAIL_MAX          = 96
PAIRFOLD_MIN_SIM           = float(os.environ.get("PAIRFOLD_MIN_SIM", "0.60"))
PAIRFOLD_MAX_GAIN_DELTA    = float(os.environ.get("PAIRFOLD_MAX_GAIN_DELTA", "0.030"))
PAIRFOLD_VECTOR_BLEND      = float(os.environ.get("PAIRFOLD_VECTOR_BLEND", "0.035"))
PAIRFOLD_DECAY             = 0.985
PAIRFOLD_COL_BINS          = 8
DUAL_PAIR_ENABLED          = False
DUAL_PAIR_SPLIT            = [0.62, 0.58]
DETERMINISTIC_REBUILD_ENABLED    = True
DETERMINISTIC_REBUILD_ACCEPT_EPS = float(os.environ.get("DETERMINISTIC_REBUILD_ACCEPT_EPS", "0.004"))

# ── v10+D 新增：迭代局部搜索 ─────────────────────────────────────────────────
ILS_ENABLED     = os.environ.get("ILS_ENABLED", "1") != "0"
# 迭代轮数：每轮对所有 rank 列各扰动一次
ILS_N_ITER      = int(os.environ.get("ILS_N_ITER", "6"))
# 初始扰动步长（相对于 B 列范数的比例）
ILS_STEP_INIT   = float(os.environ.get("ILS_STEP_INIT", "0.025"))
# 每轮步长衰减系数
ILS_STEP_DECAY  = float(os.environ.get("ILS_STEP_DECAY", "0.75"))
# 是否对重建 accepted 的层也触发（False = 只对 rejected 层触发）
ILS_ON_ACCEPTED = os.environ.get("ILS_ON_ACCEPTED", "0") != "0"

GAIN_DISTRIBUTION = "v10d_iterative_local_search"

# Validate
assert FORCED_FUSED_RANK > 0
assert 1.0 <= SVD_ENERGY_GAIN_CAP <= 1.25
assert 1.0 <= ROW_NORM_GAIN_CAP <= 1.25
assert 0.0 <= PAIRFOLD_MIN_SIM <= 1.0
assert 0.0 <= PAIRFOLD_MAX_GAIN_DELTA <= 0.10
assert 0.0 <= PAIRFOLD_VECTOR_BLEND <= 0.20
assert -0.05 <= DETERMINISTIC_REBUILD_ACCEPT_EPS <= 0.05
assert 0 <= ILS_N_ITER <= 20
assert 0.0 < ILS_STEP_INIT <= 0.2
assert 0.0 < ILS_STEP_DECAY <= 1.0

# ─── Ledger ──────────────────────────────────────────────────────────────────
class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        beta = beta or {}
        gamma = gamma or {}
        basis = json.dumps({"alpha": alpha, "src": str(src), "dst": str(dst),
                            "op": str(op), "beta": beta, "gamma": gamma},
                           sort_keys=True, default=str)
        event = {"alpha": str(alpha), "source": str(src), "dest": str(dst),
                 "op": str(op), "beta": beta, "gamma": gamma,
                 "verification_hash": hashlib.sha256(basis.encode()).hexdigest()[:16]}
        self.events.append(event)
        self.counts[(event["alpha"], event["op"])] += 1

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger]  {alpha}:{op}={count}")

    def markdown(self) -> str:
        lines = ["# GlyphMatics v10+D Transport Ledger", "",
                 f"- SVD_ENERGY_GAIN_CAP: {SVD_ENERGY_GAIN_CAP}",
                 f"- ROW_NORM_GAIN_CAP: {ROW_NORM_GAIN_CAP}",
                 f"- PAIRFOLD_MIN_SIM: {PAIRFOLD_MIN_SIM}",
                 f"- PAIRFOLD_MAX_GAIN_DELTA: {PAIRFOLD_MAX_GAIN_DELTA}",
                 f"- PAIRFOLD_VECTOR_BLEND: {PAIRFOLD_VECTOR_BLEND}",
                 f"- DETERMINISTIC_REBUILD_ACCEPT_EPS: {DETERMINISTIC_REBUILD_ACCEPT_EPS}",
                 f"- ILS_ENABLED: {ILS_ENABLED}",
                 f"- ILS_N_ITER: {ILS_N_ITER}",
                 f"- ILS_STEP_INIT: {ILS_STEP_INIT}",
                 f"- ILS_STEP_DECAY: {ILS_STEP_DECAY}",
                 f"- ILS_ON_ACCEPTED: {ILS_ON_ACCEPTED}",
                 "", "## Event summary", "", "| alpha | op | count |", "|---|---|---:|",
                ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")
        return "\n".join(lines) + "\n"

GLYPH_LEDGER = GlyphmaticTransportLedger()

# ─── Numeric helpers（与 v10 完全一致）────────────────────────────────────────
def _safe_unit(x, dim=None, eps=1e-12):
    if dim is None:
        return x / torch.linalg.vector_norm(x).clamp_min(eps)
    return x / torch.linalg.vector_norm(x, dim=dim, keepdim=True).clamp_min(eps)

def _make_even_blocks(length, count, prefix):
    count = max(1, min(int(count), int(length)))
    blocks = []
    for i in range(count):
        s = int(round(i * length / count))
        e = int(round((i + 1) * length / count))
        if e > s:
            blocks.append((s, e, f"{prefix}{i}"))
    return blocks

def _make_row_blocks(row_count, component_slices=None):
    blocks = []
    if component_slices:
        for row_start, row_end, _rank, name in component_slices:
            row_start = max(0, min(int(row_start), int(row_count)))
            row_end = max(row_start, min(int(row_end), int(row_count)))
            if row_end > row_start:
                blocks.append((row_start, row_end, str(name)))
    if not blocks:
        blocks = _make_even_blocks(row_count, min(8, row_count), "rowbin")
    return blocks

def _block_energy(vec, blocks):
    vals = []
    sq = vec.float() ** 2
    for start, end, _ in blocks:
        vals.append(sq[start:end].sum())
    out = torch.stack(vals) if vals else torch.ones(1, device=vec.device)
    return _safe_unit(out.float())

def _direction_signature(u_col, vh_row, row_blocks, col_bins):
    col_blocks = _make_even_blocks(int(vh_row.numel()), max(1, min(col_bins, int(vh_row.numel()))), "c")
    row_sig = _block_energy(u_col, row_blocks)
    col_sig = _block_energy(vh_row, col_blocks)
    u_abs, v_abs = u_col.float().abs(), vh_row.float().abs()
    moments = _safe_unit(torch.tensor([float(u_abs.max()), float(v_abs.max()),
                                       float(u_abs.mean()), float(v_abs.mean())],
                                      device=u_col.device))
    return _safe_unit(torch.cat([row_sig, col_sig, moments]).float())

def _rank_pair_groups(rank, pair_width):
    groups = []
    start = 0
    while start < rank:
        end = min(rank, start + pair_width)
        groups.append(list(range(start, end)))
        start = end
    return groups

# ─── PairFold transport（与 v10 完全一致）─────────────────────────────────────
def _pairfold_transport(U, S, Vh, rank, gain_vec, row_blocks):
    device = U.device
    rank = int(rank)
    tail_available = max(0, int(S.numel()) - rank)
    if (not PAIRFOLD_ENABLED) or rank <= 0 or tail_available <= 0 or PAIRFOLD_TAIL_MAX <= 0:
        return U[:, :rank].contiguous(), Vh[:rank, :].contiguous(), gain_vec.contiguous(), {"pairfold_mode": "disabled"}

    U_k = U[:, :rank].clone()
    Vh_k = Vh[:rank, :].clone()
    S_k = S[:rank]
    gain_in = gain_vec.clone()

    head_sigs = torch.stack([
        _direction_signature(U[:, i], Vh[i, :], row_blocks, PAIRFOLD_COL_BINS)
        for i in range(rank)
    ])

    groups = _rank_pair_groups(rank, PAIRFOLD_PAIR_WIDTH)
    group_sigs = []
    for group in groups:
        idx = torch.tensor(group, device=device, dtype=torch.long)
        w = S_k[idx].float().clamp_min(1e-12)
        w = w / w.sum().clamp_min(1e-12)
        sig = (head_sigs[idx] * w.unsqueeze(1)).sum(dim=0)
        group_sigs.append(_safe_unit(sig))
    group_sigs = torch.stack(group_sigs)

    head_priority = torch.zeros(rank, device=device, dtype=torch.float32)
    u_acc = torch.zeros_like(U_k.float())
    v_acc = torch.zeros_like(Vh_k.float())

    tail_count = min(tail_available, int(PAIRFOLD_TAIL_MAX))
    assignments = 0
    sim_sum = 0.0

    for local_j in range(tail_count):
        j = rank + local_j
        tail_sig = _direction_signature(U[:, j], Vh[j, :], row_blocks, PAIRFOLD_COL_BINS)
        sims = group_sigs @ tail_sig
        best_group = int(torch.argmax(sims).detach().cpu())
        best_sim = float(sims[best_group].detach().cpu())
        if best_sim < PAIRFOLD_MIN_SIM:
            continue

        decay = float(PAIRFOLD_DECAY ** local_j)
        rel_tail = float((S[j] / S_k.mean().clamp_min(1e-12)).detach().cpu())
        priority = max(0.0, best_sim * decay * rel_tail)
        if priority <= 0:
            continue

        assignments += 1
        sim_sum += best_sim
        group = groups[best_group]
        idx = torch.tensor(group, device=device, dtype=torch.long)
        local_sims = (head_sigs[idx] @ tail_sig).clamp_min(0.0)
        local_weights = local_sims + S_k[idx].float() / S_k[idx].float().sum().clamp_min(1e-12)
        local_weights = local_weights / local_weights.sum().clamp_min(1e-12)

        for pos, head_idx in enumerate(group):
            head_priority[head_idx] += priority * float(local_weights[pos].detach().cpu())
            if PAIRFOLD_VECTOR_BLEND > 0:
                blend = min(PAIRFOLD_VECTOR_BLEND, PAIRFOLD_VECTOR_BLEND * best_sim * rel_tail) * float(local_weights[pos].detach().cpu())
                u_acc[:, head_idx] += float(blend) * U[:, j].float()
                v_acc[head_idx, :] += float(blend) * Vh[j, :].float()

    if assignments == 0 or float(head_priority.sum()) <= 0:
        return U_k.contiguous(), Vh_k.contiguous(), gain_in.contiguous(), {"pairfold_mode": "no_match"}

    priority = head_priority / head_priority.mean().clamp_min(1e-12)
    delta = (priority - 1.0).clamp(-1.0, 1.0) * float(PAIRFOLD_MAX_GAIN_DELTA)
    gain_out = gain_in * (1.0 + delta)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    target_energy = torch.sqrt(torch.sum((S_k * gain_in) ** 2)).clamp_min(1e-12)
    current_energy = torch.sqrt(torch.sum((S_k * gain_out) ** 2)).clamp_min(1e-12)
    gain_out = torch.clamp(gain_out * (target_energy / current_energy), min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    if PAIRFOLD_VECTOR_BLEND > 0:
        U_k = _safe_unit(U_k.float() + u_acc, dim=0).to(U.dtype)
        Vh_k = _safe_unit(Vh_k.float() + v_acc, dim=1).to(Vh.dtype)

    return U_k.contiguous(), Vh_k.contiguous(), gain_out.contiguous(), {
        "pairfold_mode": "residual_matched",
        "pairfold_assignments": int(assignments),
        "pairfold_avg_sim": float(sim_sum / max(1, assignments)),
        "pairfold_gain_mean": float(gain_out.mean().detach().cpu()),
    }

# ─── RowGuard（与 v10 完全一致）──────────────────────────────────────────────
def _apply_row_norm_guard(B_new, A_new, delta):
    if not ROW_NORM_GUARD_ENABLED:
        return B_new.contiguous(), {"row_norm_guard_enabled": False}
    with torch.no_grad():
        recon = B_new.float() @ A_new.float()
        orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
        new_row = torch.linalg.vector_norm(recon, ord=2, dim=1).clamp_min(1e-12)
        row_scale = torch.minimum(torch.ones_like(new_row), orig_row * float(ROW_NORM_GAIN_CAP) / new_row)
        clipped = row_scale < 0.999
        B_guarded = (B_new.float() * row_scale.unsqueeze(1)).to(B_new.dtype).contiguous()
    return B_guarded, {"row_norm_guard_enabled": True, "row_clip_fraction": float(clipped.float().mean())}

# ─── Rebuild objective（与 v10 完全一致）─────────────────────────────────────
def _rebuild_objective(B_new, A_new, delta):
    recon = B_new.float() @ A_new.float()
    delta_norm = torch.linalg.vector_norm(delta.float()).clamp_min(1e-12)
    residual = torch.linalg.vector_norm((delta.float() - recon).float()) / delta_norm
    orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(recon.float(), ord=2, dim=1).clamp_min(1e-12)
    overshoot = torch.clamp(new_row / orig_row - float(ROW_NORM_GAIN_CAP), min=0.0).mean()
    return residual + 0.05 * overshoot, residual, overshoot

def _factorize_balanced_from_basis(U_basis, core, V_basis):
    P, Sc, Qh = torch.linalg.svd(core.float(), full_matrices=False)
    Sc = Sc.float().clamp_min(0.0)
    sroot = torch.sqrt(Sc.clamp_min(1e-12))
    B_new = (U_basis.float() @ P.float()) * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * (Qh.float() @ V_basis.float().T)
    return B_new.contiguous(), A_new.contiguous(), Sc.contiguous()

def _phase2_rebuild_fallback(U_k, Vh_k, S_k, gain_vec, delta, baseline_B, baseline_A, base_obj):
    try:
        Q_left, _ = torch.linalg.qr(U_k.float(), mode="reduced")
        Q_right, _ = torch.linalg.qr(Vh_k.float().T, mode="reduced")
        core = Q_left.T @ delta.float() @ Q_right
        B_core, A_core, Sc_core = _factorize_balanced_from_basis(Q_left, core, Q_right)

        target_energy = torch.sqrt(torch.sum((S_k.float() * gain_vec.float() * 0.92) ** 2)).clamp_min(1e-12)
        core_energy = torch.sqrt(torch.sum(Sc_core.float() ** 2)).clamp_min(1e-12)
        energy_scale = (target_energy / core_energy).clamp(0.25, 4.0)
        if torch.isfinite(energy_scale):
            scale_root = torch.sqrt(energy_scale)
            B_core = B_core * scale_root
            A_core = A_core * scale_root

        B_core, rg_stats = _apply_row_norm_guard(B_core, A_core, delta)
        cand_obj, _, _ = _rebuild_objective(B_core, A_core, delta)

        if cand_obj <= base_obj * (1.0 + DETERMINISTIC_REBUILD_ACCEPT_EPS):
            return B_core.contiguous(), A_core.contiguous(), rg_stats, True
    except Exception:
        pass
    return baseline_B.contiguous(), baseline_A.contiguous(), {"row_norm_guard_enabled": False}, False

def _deterministic_subspace_rebuild(U_k, Vh_k, S_k, gain_vec, delta, baseline_B, baseline_A, baseline_rg):
    if not DETERMINISTIC_REBUILD_ENABLED:
        return baseline_B.contiguous(), baseline_A.contiguous(), {"mode": "disabled"}, baseline_rg
    if U_k.numel() == 0 or Vh_k.numel() == 0:
        return baseline_B.contiguous(), baseline_A.contiguous(), {"mode": "empty_basis"}, baseline_rg

    with torch.no_grad():
        base_obj, _, _ = _rebuild_objective(baseline_B, baseline_A, delta)

        Q_left, _ = torch.linalg.qr(U_k.float(), mode="reduced")
        Q_right, _ = torch.linalg.qr(Vh_k.float().T, mode="reduced")
        core = Q_left.T @ delta.float() @ Q_right
        B_core, A_core, Sc_core = _factorize_balanced_from_basis(Q_left, core, Q_right)

        target_energy = torch.sqrt(torch.sum((S_k.float() * gain_vec.float()) ** 2)).clamp_min(1e-12)
        core_energy = torch.sqrt(torch.sum(Sc_core.float() ** 2)).clamp_min(1e-12)
        energy_scale = (target_energy / core_energy).clamp(0.25, 4.0)
        if torch.isfinite(energy_scale):
            scale_root = torch.sqrt(energy_scale)
            B_core = B_core * scale_root
            A_core = A_core * scale_root

        B_core, rebuild_rg = _apply_row_norm_guard(B_core, A_core, delta)
        cand_obj, _, _ = _rebuild_objective(B_core, A_core, delta)
        accept = bool(cand_obj <= base_obj * (1.0 + DETERMINISTIC_REBUILD_ACCEPT_EPS))

    if accept:
        return B_core.contiguous(), A_core.contiguous(), {
            "mode": "accepted_subspace_core",
            "accepted": True,
            "base_obj": float(base_obj), "cand_obj": float(cand_obj)
        }, rebuild_rg

    B_p2, A_p2, rg_p2, p2_ok = _phase2_rebuild_fallback(
        U_k, Vh_k, S_k, gain_vec, delta, baseline_B, baseline_A, base_obj
    )
    if p2_ok:
        return B_p2, A_p2, {
            "mode": "accepted_phase2_reduced_gain",
            "accepted": True,
            "base_obj": float(base_obj), "cand_obj": float(cand_obj)
        }, rg_p2

    return baseline_B.contiguous(), baseline_A.contiguous(), {
        "mode": "rejected_kept_pairfold_diagonal",
        "accepted": False,
        "base_obj": float(base_obj), "cand_obj": float(cand_obj)
    }, baseline_rg

# ════════════════════════════════════════════════════════════════════════════
# v10+D 核心新增：迭代局部搜索
# ════════════════════════════════════════════════════════════════════════════

def _iterative_local_search(
    B_init: torch.Tensor,
    A_init: torch.Tensor,
    delta: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    """
    对 B 的每一列逐一施加随机扰动，接受降低目标函数的扰动。
    每次接受扰动后用 lstsq 重新求最优 A，保持 delta 近似不变。

    关键设计：
    - 步长相对于每列的 L2 范数（scale-free）
    - 每轮步长按 ILS_STEP_DECAY 衰减（粗 → 细搜索）
    - RowGuard 在每次接受后重新应用，确保不违反行范数约束
    - 全程 no_grad，不触发 autograd

    返回：
      (B_out, A_out, stats)
    """
    if not ILS_ENABLED or ILS_N_ITER <= 0:
        return B_init.contiguous(), A_init.contiguous(), {"ils_enabled": False}

    with torch.no_grad():
        B = B_init.float().clone()
        A = A_init.float().clone()
        delta_f = delta.float()

        best_obj, _, _ = _rebuild_objective(B, A, delta_f)
        obj_init = float(best_obj)

        total_accepts = 0
        total_tries = 0
        step = ILS_STEP_INIT

        # 固定随机种子保证可复现（用 delta 的 hash 作为种子）
        seed = int(delta_f.abs().sum().item() * 1e4) % (2 ** 31)
        rng = torch.Generator(device=B.device)
        rng.manual_seed(seed)

        rank = B.shape[1]

        for iter_idx in range(ILS_N_ITER):
            accepts_this_iter = 0

            # 打乱列顺序，避免固定顺序带来的偏差
            col_order = torch.randperm(rank, generator=rng).tolist()

            for col_idx in col_order:
                total_tries += 1
                col_norm = float(torch.linalg.vector_norm(B[:, col_idx]).clamp_min(1e-12))

                # 随机扰动方向（单位向量），步长相对于列范数
                noise = torch.randn(B.shape[0], generator=rng, device=B.device)
                noise = noise / noise.norm().clamp_min(1e-12)
                noise = noise * col_norm * step

                B_try = B.clone()
                B_try[:, col_idx] = B_try[:, col_idx] + noise

                # lstsq 求最优 A：min ||B_try @ A - delta||_F
                # torch.linalg.lstsq 在 rank-32、大矩阵上约 1~3ms
                try:
                    result = torch.linalg.lstsq(B_try, delta_f, driver="gelsd")
                    A_try = result.solution
                except Exception:
                    continue

                if not torch.isfinite(A_try).all():
                    continue

                # RowGuard：确保候选解不违反行范数约束
                B_try_g, _ = _apply_row_norm_guard(B_try, A_try, delta_f)

                cand_obj, _, _ = _rebuild_objective(B_try_g, A_try, delta_f)

                if cand_obj < best_obj:
                    B = B_try_g
                    A = A_try
                    best_obj = cand_obj
                    total_accepts += 1
                    accepts_this_iter += 1

            # 步长衰减
            step *= ILS_STEP_DECAY

            # 早停：连续两轮无改进
            if accepts_this_iter == 0 and iter_idx >= 1:
                break

        obj_final = float(best_obj)

    return (
        B.to(B_init.dtype).contiguous(),
        A.to(A_init.dtype).contiguous(),
        {
            "ils_enabled": True,
            "ils_obj_init": round(obj_init, 6),
            "ils_obj_final": round(obj_final, 6),
            "ils_improvement": round(obj_init - obj_final, 6),
            "ils_accept_rate": round(total_accepts / max(total_tries, 1), 4),
            "ils_total_accepts": total_accepts,
        }
    )

# ─── Main compression entry point（v10+D：在 rebuild 后追加 ILS）──────────────
def _compress_lora_pair_to_rank(B, A_mat, rank, component_slices=None):
    if B.ndim != 2 or A_mat.ndim != 2:
        raise ValueError(f"Expected 2D LoRA matrices, got B={tuple(B.shape)}, A={tuple(A_mat.shape)}")
    if B.shape[1] != A_mat.shape[0]:
        raise ValueError(f"LoRA inner rank mismatch")

    delta = B.float() @ A_mat.float()
    if not torch.isfinite(delta).all():
        raise FloatingPointError("Delta contains non-finite values")

    max_rank = min(delta.shape)
    rank = min(int(rank), int(max_rank))

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    S_k = S[:rank]
    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)
    raw_gain = (full_energy / kept_energy).clamp(1.0, float(SVD_ENERGY_GAIN_CAP))
    gain_vec = torch.full_like(S_k, float(raw_gain))

    row_blocks = _make_row_blocks(int(delta.shape[0]), component_slices=component_slices)
    U_k, Vh_k, gain_vec, pairfold_stats = _pairfold_transport(
        U=U, S=S, Vh=Vh, rank=rank, gain_vec=gain_vec, row_blocks=row_blocks
    )

    sroot_balanced = torch.sqrt(S_k * gain_vec)
    B_baseline = U_k * sroot_balanced.unsqueeze(0)
    A_baseline = sroot_balanced.unsqueeze(1) * Vh_k
    B_baseline, baseline_rg = _apply_row_norm_guard(B_baseline, A_baseline, delta)

    B_new, A_new, rebuild_stats, row_guard_stats = _deterministic_subspace_rebuild(
        U_k=U_k, Vh_k=Vh_k, S_k=S_k, gain_vec=gain_vec, delta=delta,
        baseline_B=B_baseline, baseline_A=A_baseline, baseline_rg=baseline_rg
    )

    # ── v10+D 新增：ILS 后处理 ───────────────────────────────────────────────
    # 触发条件：rejected 层始终触发；accepted 层仅在 ILS_ON_ACCEPTED=True 时触发
    rebuild_accepted = rebuild_stats.get("accepted", False)
    should_run_ils = ILS_ENABLED and (not rebuild_accepted or ILS_ON_ACCEPTED)

    if should_run_ils:
        B_new, A_new, ils_stats = _iterative_local_search(B_new, A_new, delta)
    else:
        ils_stats = {"ils_enabled": False, "ils_skipped_reason": "rebuild_accepted"}
    # ─────────────────────────────────────────────────────────────────────────

    if not torch.isfinite(B_new).all() or not torch.isfinite(A_new).all():
        raise FloatingPointError("Compressed factors contain non-finite values")

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "singular_mass_kept": float((S_k.sum() / total_mass).detach().cpu()),
        "energy_kept_ratio": float((kept_energy / full_energy).detach().cpu()),
        "raw_gain": float(raw_gain.detach().cpu()),
        **pairfold_stats,
        **rebuild_stats,
        **row_guard_stats,
        **ils_stats,
    }

# ─── Tinker monkey-patch（与 v10 完全一致）───────────────────────────────────
def patched_merge_fused_projections(
    fused_model_key, adapter_layer_prefix, components,
    model_state_shapes, peft_weights, target_modules, profile
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    if component_order is None:
        raise RuntimeError(f"No fused projection component order for {fused_target_name!r}")

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts, comp_slices = [], []
    merged_rank, row_offset = 0, 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f"Missing component {comp_name!r}")
        lora_A, lora_B = comp_by_name[comp_name]
        r, out_dim = int(lora_A.shape[0]), int(lora_B.shape[0])
        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(fused_out_dim, merged_rank,
                                dtype=merged_lora_A.dtype, device=merged_lora_A.device)
    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = int(merged_rank)
    compression_stats = {"preservation": "exact", "rank_in": int(merged_rank)}

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, svd_stats = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK,
            component_slices=comp_slices
        )
        final_rank = int(merged_lora_A.shape[0])
        compression_stats = {**svd_stats, "preservation": "rank32_v10d_ils"}

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op="fused_projection_v10d",
        alpha="mamba_or_fused_projection",
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


if not hasattr(A, "_merge_fused_projections"):
    raise RuntimeError("tinker_cookbook._merge_fused_projections not found")
A._merge_fused_projections = patched_merge_fused_projections

print("[GlyphMatics v10+D] patched:", A._merge_fused_projections.__name__)
print(f"[GlyphMatics v10+D] ILS: enabled={ILS_ENABLED} n_iter={ILS_N_ITER} "
      f"step_init={ILS_STEP_INIT} decay={ILS_STEP_DECAY} on_accepted={ILS_ON_ACCEPTED}")
print(f"[GlyphMatics v10+D] SVD_ENERGY_GAIN_CAP={SVD_ENERGY_GAIN_CAP} ROW_NORM_GAIN_CAP={ROW_NORM_GAIN_CAP}")

# %% [code]
# ─── Self-tests ──────────────────────────────────────────────────────────────
import torch
print("[SelfTest] running v10+D tests")
_gen = torch.Generator(device="cpu").manual_seed(42)

# --- Test 1: ILS 改进目标函数 ---
B_test = torch.randn(48, 8, generator=_gen, dtype=torch.float32)
A_test = torch.randn(8, 40, generator=_gen, dtype=torch.float32)
delta_test = B_test @ A_test

# 故意引入误差
B_noisy = B_test + torch.randn_like(B_test, generator=_gen) * 0.3
A_noisy = A_test + torch.randn_like(A_test, generator=_gen) * 0.3

obj_before, _, _ = _rebuild_objective(B_noisy, A_noisy, delta_test)
B_out, A_out, ils_stats = _iterative_local_search(B_noisy, A_noisy, delta_test)
obj_after, _, _ = _rebuild_objective(B_out, A_out, delta_test)

print(f"[SelfTest] ILS: obj {float(obj_before):.6f} → {float(obj_after):.6f} "
      f"improvement={ils_stats['ils_improvement']:.6f} "
      f"accepts={ils_stats['ils_total_accepts']}")
assert float(obj_after) <= float(obj_before) + 1e-6, "ILS 不应使目标函数变差"
assert torch.isfinite(B_out).all() and torch.isfinite(A_out).all()

# --- Test 2: 端到端压缩 ---
B_test2 = torch.randn(48, 24, generator=_gen, dtype=torch.float32)
A_test2 = torch.randn(24, 40, generator=_gen, dtype=torch.float32)
cs_test = [(0, 12, 8, "q_proj"), (12, 24, 8, "k_proj"), (24, 36, 4, "v_proj"), (36, 48, 4, "gate")]

B_c, A_c, stats = _compress_lora_pair_to_rank(B_test2, A_test2, rank=8, component_slices=cs_test)

assert tuple(B_c.shape) == (48, 8), tuple(B_c.shape)
assert tuple(A_c.shape) == (8, 40), tuple(A_c.shape)
assert torch.isfinite(B_c).all() and torch.isfinite(A_c).all()
assert stats["rank_out"] == 8
print(f"[SelfTest] compress: rebuild={stats.get('mode')} "
      f"ils_improvement={stats.get('ils_improvement', 'skipped')}")

# --- Test 3: RowGuard ---
if ROW_NORM_GUARD_ENABLED:
    orig_delta = B_test2 @ A_test2
    new_delta = B_c.float() @ A_c.float()
    orig_row = torch.linalg.vector_norm(orig_delta, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(new_delta, dim=1).clamp_min(1e-12)
    worst = float((new_row / orig_row).max())
    assert worst <= ROW_NORM_GAIN_CAP + 2e-4, f"RowGuard violation: {worst:.4f}"
    print(f"[SelfTest] RowGuard OK: worst_ratio={worst:.4f}")

print("[SelfTest] ALL PASSED")

# %% [code]
# ─── Build adapter ───────────────────────────────────────────────────────────
from pathlib import Path
import shutil, time, statistics
from tinker_cookbook import weights

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

print("[Build] adapter_path:", ADAPTER_PATH)
print("[Build] base_model:", BASE_MODEL_PATH)
print("[Build] output:", OUTPUT_DIR)

t0 = time.time()
weights.build_lora_adapter(
    base_model=str(BASE_MODEL_PATH),
    adapter_path=str(ADAPTER_PATH),
    output_path=str(OUTPUT_DIR),
)
elapsed = time.time() - t0
print(f"[Build] completed in {elapsed/60:.2f} min")

missing = [n for n in ["adapter_config.json", "adapter_model.safetensors"]
           if not (OUTPUT_DIR / n).exists()]
if missing:
    raise FileNotFoundError(f"Build did not produce: {missing}")

(OUTPUT_DIR / "checkpoint_complete").write_text("ok\n", encoding="utf-8")

GLYPH_LEDGER.print_summary()
(OUTPUT_DIR / "GLYPHMATICS_TRANSPORT_LEDGER.md").write_text(
    GLYPH_LEDGER.markdown(), encoding="utf-8"
)

# 统计 ILS 效果
improvements, accept_rates = [], []
skipped = 0
for ev in GLYPH_LEDGER.events:
    g = ev.get("gamma", {})
    imp = g.get("ils_improvement")
    ar = g.get("ils_accept_rate")
    if imp is not None:
        improvements.append(imp)
        accept_rates.append(ar)
    else:
        skipped += 1

if improvements:
    print(f"[Build] ILS stats over {len(improvements)} layers (skipped={skipped}):")
    print(f"  improvement: min={min(improvements):.6f}  max={max(improvements):.6f}  "
          f"mean={statistics.mean(improvements):.6f}")
    print(f"  accept_rate: min={min(accept_rates):.4f}  max={max(accept_rates):.4f}  "
          f"mean={statistics.mean(accept_rates):.4f}")

readme_path = OUTPUT_DIR / "README.md"
readme_content = f"""# Nemotron Adapter — GlyphMatics v10+D

Iterative Local Search + Stacked Gain + Dual-Phase Deterministic Rebuild

## Config
- SVD_ENERGY_GAIN_CAP: {SVD_ENERGY_GAIN_CAP}
- ROW_NORM_GAIN_CAP: {ROW_NORM_GAIN_CAP}
- PAIRFOLD_MIN_SIM: {PAIRFOLD_MIN_SIM}
- PAIRFOLD_MAX_GAIN_DELTA: {PAIRFOLD_MAX_GAIN_DELTA}
- PAIRFOLD_VECTOR_BLEND: {PAIRFOLD_VECTOR_BLEND}
- DETERMINISTIC_REBUILD_ACCEPT_EPS: {DETERMINISTIC_REBUILD_ACCEPT_EPS}
- FORCED_FUSED_RANK: {FORCED_FUSED_RANK}
- ILS_ENABLED: {ILS_ENABLED}
- ILS_N_ITER: {ILS_N_ITER}
- ILS_STEP_INIT: {ILS_STEP_INIT}
- ILS_STEP_DECAY: {ILS_STEP_DECAY}
- ILS_ON_ACCEPTED: {ILS_ON_ACCEPTED}
"""
if readme_path.exists():
    with open(readme_path, "a") as f:
        f.write("\n" + readme_content)
else:
    readme_path.write_text(readme_content)

print("[Build] output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:50s} {p.stat().st_size:>12,}")

# %% [code]
# ─── Validate and create submission.zip ──────────────────────────────────────
import zipfile, hashlib

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

required = ["adapter_config.json", "adapter_model.safetensors", "README.md", "checkpoint_complete"]
missing = [n for n in required if not (OUTPUT_DIR / n).exists()]
if missing:
    raise FileNotFoundError(f"Missing: {missing}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in required:
        zf.write(OUTPUT_DIR / name, arcname=name)

h = hashlib.sha256()
with ZIP_PATH.open("rb") as f:
    while chunk := f.read(1 << 20):
        h.update(chunk)

with zipfile.ZipFile(ZIP_PATH) as zf:
    bad = zf.testzip()
    names = zf.namelist()

print("[Zip] wrote:", ZIP_PATH)
print("[Zip] size:", ZIP_PATH.stat().st_size)
print("[Zip] sha256:", h.hexdigest())
print("[Zip] contents:", names)
assert bad is None, f"Zip integrity error: {bad}"
assert names == required
print("[Zip] validation PASSED")

# %% [code]
!ls -lah /kaggle/working/submission.zip /kaggle/working/nemotron-adapter-ready-to-submit

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models
[Tinker] scanning wheel folders...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker]  tinker-cookbook: 0.3.0
[Tinker]

MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[Build] completed in 3.28 min
[GlyphMatics ledger] events: 23
[GlyphMatics ledger]  mamba_or_fused_projection:fused_projection_v10d=23
[Build] output files:
  GLYPHMATICS_TRANSPORT_LEDGER.md                             447
  README.md                                                   439
  adapter_config.json                                         618
  adapter_model.safetensors                          3,554,384,888
  checkpoint_complete                                           3
[Zip] wrote: /kaggle/working/submission.zip
[Zip] size: 3270301044
[Zip] sha256: 6debbca4494dd139f44545d98956f089206d86ee86bbba963e2604359c89b65a
[Zip] contents: ['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'checkpoint_complete']
[Zip] validation PASSED
-rw-r--r-- 1 root root 3.1G Jun  1 03:24 /kaggle/working/submission.zip

/kaggle/working/nemotron-adapter-ready-to-submit:
total 3.4G
drwxr-xr-x 2 root root 4.0K Jun  1 03:21 .
drwxr-xr-x 3 root root 4.0K Jun  1 03:21 ..
-rw-r--r-- 1 roo